# 00 — Data Preparation

Download OpenXAI Adult Income dataset, compute per-instance SHAP values
using a pretrained logistic regression model, and save a processed CSV to `data/processed/`.

**Run this notebook once** before running the generation pipeline.
For large-scale preparation, prefer the CLI script:
```bash
python scripts/prepare_data.py
```

### Prerequisites
OpenXAI must be installed from source:
```bash
git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
cd OpenXAI && pip install -e .
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline

In [ ]:
# Verify OpenXAI is available
try:
    from openxai import LoadModel, Explainer
    from openxai.dataloader import ReturnLoaders
    print('OpenXAI imported successfully.')
except ImportError:
    print('ERROR: OpenXAI not installed.')
    print('Run: git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git && cd OpenXAI && pip install -e .')

## Configuration

In [ ]:
# Settings — adjust as needed
ML_MODEL   = 'lr'      # 'lr' (logistic regression) or 'ann' (neural network)
SPLIT      = 'test'    # 'test' or 'train'
N          = None      # Set to an integer to cap instances, None = use all
SHAP_PREFIX = 'shap_'
OUTPUT_DIR  = '../data/processed'

DATASETS = [
    {'our_name': 'adult', 'openxai_name': 'adult'},
]

## Step 1 — Load data

In [ ]:
import torch

def load_split(openxai_name, split='test', n=None):
    trainloader, testloader = ReturnLoaders(data_name=openxai_name, download=True)
    loader = testloader if split == 'test' else trainloader
    
    X_batches, y_batches = [], []
    for bx, by in loader:
        X_batches.append(bx.numpy())
        y_batches.append(by.numpy())
    
    X = np.concatenate(X_batches)
    y = np.concatenate(y_batches)
    
    # Also collect training data for SHAP background
    X_train_batches = []
    for bx, _ in trainloader:
        X_train_batches.append(bx.numpy())
    X_train = np.concatenate(X_train_batches)
    
    if n is not None:
        X, y = X[:n], y[:n]
    
    print(f'  {openxai_name}: X={X.shape}, X_train={X_train.shape}')
    return X, y, X_train

datasets_data = {}
for ds in DATASETS:
    X, y, X_train = load_split(ds['openxai_name'], SPLIT, N)
    datasets_data[ds['our_name']] = {'X': X, 'y': y, 'X_train': X_train, 'openxai_name': ds['openxai_name']}

## Step 2 — Inspect raw data

In [ ]:
for name, d in datasets_data.items():
    print(f'\n{name}')
    print(f'  Instances : {d["X"].shape[0]}')
    print(f'  Features  : {d["X"].shape[1]}')
    print(f'  Label dist: {pd.Series(d["y"]).value_counts().to_dict()}')
    print(f'  X range   : [{d["X"].min():.3f}, {d["X"].max():.3f}]')

## Step 3 — Load pretrained models and compute SHAP values

In [ ]:
from scripts.prepare_data import _get_feature_names

for name, d in datasets_data.items():
    openxai_name = d['openxai_name']
    print(f'\nProcessing {name} ({openxai_name})...')
    
    # Load model
    model = LoadModel(data_name=openxai_name, ml_model=ML_MODEL, pretrained=True)
    
    # Build tensors
    X_t      = torch.FloatTensor(d['X'])
    y_t      = torch.LongTensor(d['y'].astype(int))
    Xtrain_t = torch.FloatTensor(d['X_train'])
    
    # SHAP explainer
    explainer = Explainer(method='shap', model=model, param_dict={'x_train': Xtrain_t})
    
    try:
        shap_vals = explainer.get_explanations(X_t, y_t)
    except TypeError:
        shap_vals = explainer.get_explanations(X_t)
    
    shap_np = shap_vals.detach().numpy() if hasattr(shap_vals, 'detach') else np.array(shap_vals)
    d['shap'] = shap_np
    
    n_features = d['X'].shape[1]
    d['feature_names'] = _get_feature_names(openxai_name, n_features)
    
    print(f'  SHAP shape   : {shap_np.shape}')
    print(f'  Feature names: {d["feature_names"]}')

## Step 4 — Explore SHAP values

In [ ]:
for name, d in datasets_data.items():
    feat_names = d['feature_names']
    shap_np = d['shap']
    
    mean_abs = np.abs(shap_np).mean(axis=0)
    order = np.argsort(mean_abs)[::-1]
    
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(
        [feat_names[i] for i in order],
        mean_abs[order],
        color='steelblue', edgecolor='white'
    )
    ax.set_title(f'Mean |SHAP| — {name}', fontweight='bold')
    ax.set_ylabel('Mean |SHAP value|')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print(f'{name} — top 3 features by |SHAP|:')
    for i in order[:3]:
        print(f'  {feat_names[i]}: {mean_abs[i]:.4f}')

In [ ]:
# Inspect one instance
for name, d in datasets_data.items():
    feat_names = d['feature_names']
    row_idx = 0
    X_row = d['X'][row_idx]
    shap_row = d['shap'][row_idx]
    label = int(d['y'][row_idx])
    
    print(f'\n=== {name} | instance {row_idx} | label={label} ===')
    items = sorted(zip(feat_names, X_row, shap_row), key=lambda x: abs(x[2]), reverse=True)
    for feat, val, shap in items:
        sign = '+' if shap >= 0 else ''
        print(f'  {feat:30s}: value={val:.4f}  shap={sign}{shap:.4f}')

## Step 5 — Build and save DataFrames

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILENAMES = {
    'adult': 'adult.csv',
}

for name, d in datasets_data.items():
    feat_names = d['feature_names']
    
    df = pd.DataFrame(d['X'], columns=feat_names)
    df['label'] = d['y'].astype(int)
    
    for i, feat in enumerate(feat_names):
        df[f"{SHAP_PREFIX}{feat}"] = d['shap'][:, i]
    
    out_path = f"{OUTPUT_DIR}/{OUTPUT_FILENAMES[name]}"
    df.to_csv(out_path, index=False)
    print(f'Saved {len(df)} rows → {out_path}')
    print(f'  Columns: {list(df.columns)}')

## Step 6 — Validate output

In [ ]:
from scripts.prepare_data import validate_output
from pathlib import Path

for name in datasets_data:
    validate_output(Path(f"{OUTPUT_DIR}/{OUTPUT_FILENAMES[name]}"), SHAP_PREFIX)

## Done

Processed CSVs are in `data/processed/`. You can now run the generation pipeline:
```bash
python scripts/run_generation.py --dry-run --dataset adult --n 5
```